In [ ]:
# Importar bibliotecas

import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Configurações de caminhos

MODEL_PATH = "C:/Matheus/tech_challenge_fase4/models/lstm_close_price.keras"
SCALER_PATH = "C:/Matheus/tech_challenge_fase4/models/scaler.pkl"
TEST_DATA_PATH = "C:/Matheus/tech_challenge_fase4/data/raw/MSFT.csv"

In [ ]:
# Carregar modelo e scaler

import joblib

model = load_model(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
print("Min:", scaler.min_)
print("Scale:", scaler.scale_)
print("Data min original:", scaler.data_min_)
print("Data max original:", scaler.data_max_)


In [ ]:
# Carregar dados de teste

df_test = pd.read_csv(TEST_DATA_PATH)
# Ajuste a coluna de acordo com sua feature alvo
target_col = 'Close'
values = df_test[[target_col]].values

df_test = df_test.drop([0, 1]).reset_index(drop=True)
df_test[[target_col]] = df_test[[target_col]].astype(float)
values = df_test[[target_col]].values
print(values)

In [ ]:
# Pré-processamento para LSTM

# Escalar os dados
values_scaled = scaler.transform(values)
print(values_scaled)

# Ajustar shape para LSTM: (samples, timesteps, features)
X_test_scaled = values_scaled.reshape((values_scaled.shape[0], 1, values_scaled.shape[1]))

In [ ]:
# Fazer previsões

y_pred_scaled = model.predict(X_test_scaled)

# Desescalar
y_pred = scaler.inverse_transform(y_pred_scaled)
print(y_pred)

In [ ]:
# Comparar previsões com valores reais

plt.figure(figsize=(12,6))
plt.plot(df_test[target_col].values, label='Real')
plt.plot(y_pred, label='Predito')
plt.title('Previsão vs Real - LSTM')
plt.xlabel('Tempo')
plt.ylabel('Preço')
plt.legend()
plt.show()

In [ ]:
# Calcular métricas

mae = mean_absolute_error(df_test[target_col], y_pred)
rmse = np.sqrt(mean_squared_error(df_test[target_col], y_pred))

print(f"MAE: {mae:.5f}")
print(f"RMSE: {rmse:.5f}")

In [ ]:
# -----------------------------
# CMD DE NOTEBOOK: VALIDAR SCALER E PREVISÃO
# -----------------------------

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Para o modelo Keras
from tensorflow.keras.models import load_model

# -----------------------------
# 1. Carregar scaler e modelo
# -----------------------------
scaler = joblib.load(SCALER_PATH)
model = load_model(MODEL_PATH)

# -----------------------------
# 2. Preparar dados de teste
# -----------------------------
# Suponha que df_test tenha a coluna 'Close'
X_test = df_test[['Close']].values  # precisa ser 2D

# -----------------------------
# 3. Transformar dados
# -----------------------------
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# 4. Prever com o modelo
# -----------------------------
y_pred_scaled = model.predict(X_test_scaled)

# Certifique-se que está em 2D antes de inverter
if y_pred_scaled.ndim == 1:
    y_pred_scaled = y_pred_scaled.reshape(-1, 1)

# -----------------------------
# 5. Inverter escala
# -----------------------------
y_pred = scaler.inverse_transform(y_pred_scaled)

# -----------------------------
# 6. Clipping (opcional, previne extrapolação absurda)
# -----------------------------
y_pred = np.clip(y_pred, scaler.data_min_, scaler.data_max_)

# -----------------------------
# 7. Comparar com valores reais
# -----------------------------
y_true = df_test[['Close']].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

# -----------------------------
# 8. Plot (opcional)
# -----------------------------
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(y_true, label='Real')
plt.plot(y_pred, label='Previsto')
plt.title("Previsão x Real")
plt.legend()
plt.show()